# Coarse blend search with fixed class weights
Requires the completed native, improved and regional experiments. Search only on fold 0; confirm on folds 1–4. Existing trained models are reused; no new classifier is fitted.


In [ ]:
from pathlib import Path
import sys,json,hashlib
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score,classification_report
root=Path.cwd() if (Path.cwd()/'scripts').is_dir() else Path.cwd().parent
if str(root) not in sys.path:sys.path.insert(0,str(root))
from scripts.predict import WeightedFoldPipelineEnsemble

baseline=np.load(root/'reports/regional_predictions.npz')
y=baseline['y'];weights=baseline['decision_weights']
splits=list(StratifiedKFold(5,shuffle=True,random_state=42).split(np.zeros(len(y)),y))
selection=splits[0][1];confirmation=np.concatenate([v for _,v in splits[1:]])
families={}
for name,prefix in [('native','native'),('wide','improved_wide'),('regional','regional_medium')]:
    oof=np.zeros_like(baseline['oof']);test=np.zeros_like(baseline['test'])
    for fold,(_,validation) in enumerate(splits):
        saved=np.load(root/'reports'/f'{prefix}_fold_{fold}.npz')
        oof[validation]=saved['validation'];test+=saved['test']/5
    families[name]={'oof':oof,'test':test,'prefix':prefix}
np.testing.assert_allclose(.25*families['native']['oof']+.25*families['wide']['oof']+.5*families['regional']['oof'],baseline['oof'],atol=1e-12)
trials=[]
# Small predeclared grid; never optimize class decision weights here.
for regional_share in [.25,.375,.5,.625]:
    for native_share in [.125,.25,.375]:
        wide_share=1-regional_share-native_share
        if wide_share<.125:continue
        mix={'native':native_share,'wide':wide_share,'regional':regional_share}
        probability=sum(mix[k]*families[k]['oof'][selection] for k in mix)
        score=f1_score(y[selection],(probability*weights).argmax(1)+1,average='macro')
        trials.append({'mix':mix,'selection_macro_f1':score})
# Prefer the incumbent if the grid maximum is tied with it.
incumbent=next(t for t in trials if t['mix']=={'native':.25,'wide':.25,'regional':.5})
best=max([incumbent]+trials,key=lambda t:t['selection_macro_f1'])
(root/'reports/blend_frozen_selection.json').write_text(json.dumps({'trials':trials,'selected':best},indent=2))
print('Frozen selection: '+json.dumps(best),flush=True)
mix=best['mix'];oof=sum(mix[k]*families[k]['oof'] for k in mix);test_probability=sum(mix[k]*families[k]['test'] for k in mix)
pred=(oof*weights).argmax(1)+1;old=(baseline['oof']*weights).argmax(1)+1
report={'mix':mix,'cv_macro_f1':f1_score(y,pred,average='macro'),'confirmation_macro_f1':f1_score(y[confirmation],pred[confirmation],average='macro'),'baseline_cv_macro_f1':f1_score(y,old,average='macro'),'baseline_confirmation_macro_f1':f1_score(y[confirmation],old[confirmation],average='macro'),'classification_report':classification_report(y,pred,output_dict=True),'baseline_leaderboard':0.59326,'new_leaderboard':None,'protocol':'Predeclared coarse mixture grid on fold 0 only; class multipliers frozen to incumbent. Evaluate frozen mixture on folds 1-4. Repeated historical use means internal validation, not unbiased new evaluation.'}
report['confirmation_improved']=report['confirmation_macro_f1']>report['baseline_confirmation_macro_f1']
(root/'reports/blend_model_metrics.json').write_text(json.dumps(report,indent=2))
rows=[]
for fold,(_,v) in enumerate(splits):rows.append({'fold':fold,'role':'selection' if fold==0 else 'confirmation','baseline_macro_f1':f1_score(y[v],old[v],average='macro'),'macro_f1':f1_score(y[v],pred[v],average='macro')})
pd.DataFrame(rows).to_csv(root/'reports/blend_fold_scores.csv',index=False)
if report['confirmation_improved']:
    names=[];coefficients=[]
    for name,share in mix.items():
        names.extend([f"{families[name]['prefix']}_fold_{fold}.pkl" for fold in range(5)])
        coefficients.extend([share/5]*5)
    ensemble=WeightedFoldPipelineEnsemble(names,weights.tolist(),coefficients)
    joblib.dump(ensemble,root/'models/blend_ensemble.pkl')
    np.savez_compressed(root/'reports/blend_predictions.npz',oof=oof,test=test_probability,y=y,decision_weights=weights)
    test=pd.read_csv(root/'data/test.csv');sample=pd.read_csv(root/'data/sample_submission.csv')
    assert sample.building_id.is_unique and test.building_id.is_unique and set(sample.building_id)==set(test.building_id)
    mapping=pd.Series((test_probability*weights).argmax(1)+1,index=test.building_id)
    sample['damage_grade']=sample.building_id.map(mapping)
    assert not sample.isna().any().any() and sample.damage_grade.isin(range(1,6)).all()
    sample.damage_grade=sample.damage_grade.astype('int64')
    sample.to_csv(root/'submissions/submission_blend.csv',index=False)
print(json.dumps(report,indent=2),flush=True)
